# Samudra Prahari — train on the uploaded data (T2.1 / T2.3 / T3.1)
Runs on Colab GPU. Data comes from the leak-free 70/15/15 re-split materialised by
`scripts/data/materialize_resplit.py` (`_resplit/*/data.yaml`). Config: `configs/training.yaml`.

**Honesty (R4):** shipwreck/aircraft are REAL; pipe/cylinder/ghost_net are SYNTHETIC —
report separately. Segmentation ships **0 real masks** (synthetic ghost_net only) → real
UNet IoU stays AWAITING an AI4Shipwrecks download. Anomaly seabed is synthetic (sim-to-real gap).

In [ ]:
# 0. Setup
from google.colab import drive; drive.mount('/content/drive')
%cd /content
!git clone https://github.com/ASugandhan/SamudraPrahari.git || (cd SamudraPrahari && git pull)
%cd SamudraPrahari
!pip install -q -e '.[models]' ultralytics
# Place the datasets (or symlink from Drive), then rebuild the re-split:
!python scripts/data/index_samudra_v2.py --root Samudra_Prahari_Dataset_v2
!python scripts/data/materialize_resplit.py
import yaml; CFG = yaml.safe_load(open('configs/training.yaml'))

In [ ]:
# 1. T2.3 — YOLO11n detector (+ YOLOv8n comparison). Sonar-safe aug only.
from ultralytics import YOLO
d = CFG['detector']
for base in (d['model'], d['compare_with']):
    m = YOLO(base)
    m.train(data=d['data'], imgsz=d['imgsz'], epochs=d['epochs'], **d['augment'])
    print(base, '=> TEST metrics:'); print(m.val(data=d['data'], split='test').results_dict)
# export the YOLO11n run + register
YOLO(d['model']).export(format='onnx', opset=13)
!python scripts/register_model.py --onnx yolo11n.onnx --name yolo11n --trained-on 'Detection re-split (real+synth, R4)'

In [ ]:
# 2. T3.1 — segmentation. Provided data is YOLO-seg (synthetic ghost_net ONLY).
#    Real shipwreck IoU is NOT obtainable here — see AWAITING_REAL_MASKS in config.
assert CFG['segmentation'].get('AWAITING_REAL_MASKS'), 'add real AI4 masks before citing IoU'
from ultralytics import YOLO
seg = YOLO('yolo11n-seg.pt')
seg.train(data=CFG['segmentation']['data'], imgsz=640, epochs=CFG['segmentation']['epochs'])
print('SYNTHETIC ghost_net seg (report separately, R4):', seg.val(split='test').results_dict)

In [ ]:
# 3. T2.1 — convolutional autoencoder on object-free seabed (SSIM+L1).
import torch, torch.nn as nn, glob
from PIL import Image; import numpy as np
a = CFG['anomaly']
class AE(nn.Module):
    def __init__(s):
        super().__init__()
        s.enc = nn.Sequential(nn.Conv2d(1,16,3,2,1), nn.ReLU(), nn.Conv2d(16,32,3,2,1), nn.ReLU())
        s.dec = nn.Sequential(nn.ConvTranspose2d(32,16,4,2,1), nn.ReLU(),
                              nn.ConvTranspose2d(16,1,4,2,1), nn.Sigmoid())
    def forward(s,x): return s.dec(s.enc(x))
# ... train on glob(a['train_images']+'/*.jpg'); AUROC eval object-free (anomaly test) vs
#     object crops (detection test). Then export ONNX + register 'stage_a_ae'.
print('train dir:', a['train_images'], '| CAVEAT:', a['note'].strip()[:60], '...')